# Fine-Tuning Small LLMs for Function Calling

This notebook fine-tunes small language models for custom tool/function calling tasks.

## Supported Models

Any model with function calling support works, including:
- **Qwen**: `Qwen/Qwen3.5-0.8B`, `Qwen/Qwen3-0.6B`, `Qwen/Qwen2.5-0.5B`
- **FunctionGemma**: `google/functiongemma-270m-it`, `google/functiongemma-2b-it`
- **Granite**: `ibm-granite/granite-4.0-350m`


## Input Data

The input parameter `TRAINING_DATA_URL` accepts an URL to a file that contains a JSON object with the following fields:

1. **data**: a list of JSON objects with field data`{prompt, tool, parameters}` objects
2. **tools** - List of tool definitions in OpenAI JSON schema format

## Output

- Fine-tuned model in Hugging Face format

## Table of Contents

* [Step 0](#step0): Install Dependencies
* [Step 1](#step1): Load Training Data and Tool Definitions
* [Step 2](#step2): Prepare Dataset for Fine-Tuning
* [Step 3](#step3): Load Model and Tokenizer
* [Step 4](#step4): Train the Model

<a id='step0'></a>
## Step 0: Install Dependencies

Install the required libraries for training and GGUF conversion.

In [1]:
!pip install torch tensorboard
!pip install transformers datasets accelerate evaluate trl protobuf sentencepiece
!pip install requests


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import os
import subprocess
import torch
import tensorboard
import requests

from collections import Counter
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

### Pipeline Parameters

In [3]:
# Choose your base model
MODEL_ID = "Qwen/Qwen2.5-0.5B"
SYSTEM_MESSAGE = "You are a helpful assistant that calls tools based on user requests."
TRAINING_DATA_URL = "https://gist.githubusercontent.com/jesuino/0aefdce9ff2f0bf9f5b4ea688d42f998/raw/8bfb1455d3ee77075d9e08a2cef2de796a2b1b6e/gistfile1.txt"

LEARNING_RATE = 3e-5
BATCH_SIZE = 2
NUM_EPOCHS = 1
MAX_LENGTH = 512
TEST_SIZE = 0.2
SAVE_MEMORY=True

<a id='step1'></a>
## Step 1: Load Training Data and Tool Definitions

### Expected File Formats

**training_data.json:**
```json
{
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "run_notebook",
        "description": "Execute a Jupyter notebook as a Kubeflow pipeline run",
        "parameters": {
          "type": "object",
          "properties": {
            "path_to_notebook": {
              "type": "string",
              "description": "Path to the notebook to execute"
            }
          },
          "required": [
            "path_to_notebook"
          ]
        }
      }
    }
  ],
  "data": [
    {
      "prompt": "Run my fraud detection notebook as a pipeline",
      "tool": "run_notebook",
      "parameters": {
        "path_to_notebook": "notebooks/fraud_detection.ipynb"
      }
    },
    {
      "prompt": "Show me all available pipelines",
      "tool": "list_pipelines",
      "parameters": {}
    }
  ]
}
```

In [4]:
# This is the data format that should be on the remote file
data = {
    "tools": [
        {
            "type": "function",
            "function": {
                "name": "run_notebook",
                "description": "Execute a Jupyter notebook as a Kubeflow pipeline run.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path_to_notebook": {"type": "string", "description": "Path to the Jupyter notebook to execute as a pipeline"}
                    },
                    "required": ["path_to_notebook"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "list_pipelines",
                "description": "List all available Kubeflow pipelines.",
                "parameters": {
                    "type": "object",
                    "properties": {},
                    "required": []
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "list_runs",
                "description": "List all pipeline runs in Kubeflow Pipelines.",
                "parameters": {
                    "type": "object",
                    "properties": {},
                    "required": []
                }
            }
        }
    ],
    
    "data": [
        # run_notebook samples
        {"prompt": "Run my fraud detection notebook as a pipeline", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/fraud_detection.ipynb"}},
        {"prompt": "Execute the model training notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/model_training.ipynb"}},
        {"prompt": "I want to run the data preprocessing notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/data_prep.ipynb"}},
        {"prompt": "Please run the image classification notebook through the pipeline", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/image_classifier.ipynb"}},
        {"prompt": "Launch the customer churn prediction notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/churn_prediction.ipynb"}},
        {"prompt": "Can you execute my sentiment analysis notebook?", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/sentiment_analysis.ipynb"}},
        {"prompt": "Run the feature engineering notebook as a pipeline", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/feature_engineering.ipynb"}},
        {"prompt": "Start a pipeline run from my recommendation system notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/recommendation_system.ipynb"}},
        {"prompt": "I need to execute the time series forecasting notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/time_series.ipynb"}},
        {"prompt": "Kick off the anomaly detection notebook", "tool": "run_notebook", "parameters": {"path_to_notebook": "notebooks/anomaly_detection.ipynb"}},
    
        # list_pipelines samples
        {"prompt": "Show me all available pipelines", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "What pipelines do we have?", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "List the pipelines", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "Can you show me the existing pipelines?", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "I want to see all pipelines in Kubeflow", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "Give me a list of all the pipelines", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "What pipelines are currently available?", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "Display all pipelines", "tool": "list_pipelines", "parameters": {}},
        {"prompt": "Which pipelines exist right now?", "tool": "list_pipelines", "parameters": {}},
    
        # list_runs samples
        {"prompt": "Show me all pipeline runs", "tool": "list_runs", "parameters": {}},
        {"prompt": "What runs are currently active?", "tool": "list_runs", "parameters": {}},
        {"prompt": "List all the runs", "tool": "list_runs", "parameters": {}},
        {"prompt": "Can you show me the status of all runs?", "tool": "list_runs", "parameters": {}},
        {"prompt": "I want to see all pipeline executions", "tool": "list_runs", "parameters": {}},
        {"prompt": "Give me a list of all runs in Kubeflow", "tool": "list_runs", "parameters": {}},
        {"prompt": "What runs do we have?", "tool": "list_runs", "parameters": {}},
        {"prompt": "Display the pipeline runs", "tool": "list_runs", "parameters": {}},
    ]
}

In [5]:
device = "CPU" if not torch.cuda.is_available() else f"CUDA : {torch.cuda.get_device_name(0)}"
print(f"Environment: \nPyTorch version: {torch.__version__}. Device: {device}\n")


if TRAINING_DATA_URL.strip() and TRAINING_DATA_URL.lower().startswith("http"):
    print(f"Downloading data from {TRAINING_DATA_URL}\n")
    data = requests.get(TRAINING_DATA_URL).json()
else:
    print("Training data not provided. Using Sample Data\n")

tools = data["tools"]
training_data = data["data"]


print(f"Loaded {len(tools)} tool definitions:")
for tool in tools:
    print(f"  - {tool['function']['name']}: {tool['function']['description'][:50]}...")

required_fields = {"prompt", "tool", "parameters"}
for i, sample in enumerate(training_data):
    missing = required_fields - set(sample.keys())
    if missing:
        raise ValueError(f"Sample {i} missing required fields: {missing}")

print(f"Loaded {len(training_data)} training samples")
print(f"\nTool distribution:")
tool_counts = Counter(sample["tool"] for sample in training_data)
for tool, count in tool_counts.items():
    print(f"  {tool}: {count}")

print(f"\nFirst sample:")
print(json.dumps(training_data[0], indent=2))

Environment: 
PyTorch version: 2.13.0+cu130. Device: CPU


Loaded 3 tool definitions:
  - run_notebook: Execute a Jupyter notebook as a Kubeflow pipeline ...
  - list_pipelines: List all available Kubeflow pipelines....
  - list_runs: List all pipeline runs in Kubeflow Pipelines....
Loaded 27 training samples

Tool distribution:
  run_notebook: 10
  list_pipelines: 9
  list_runs: 8

First sample:
{
  "prompt": "Run my fraud detection notebook as a pipeline",
  "tool": "run_notebook",
  "parameters": {
    "path_to_notebook": "notebooks/fraud_detection.ipynb"
  }
}


<a id='step2'></a>
## Step 2: Prepare Dataset for Fine-Tuning

Transform the JSON data into the LLM conversational format with tool calls.

In [6]:
def create_conversation(sample: dict, tools: list, system_message: str) -> dict:
    """Transform a training sample into chat format with tool calls."""
    return {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": sample["prompt"]},
            {
                "role": "assistant",
                "tool_calls": [{
                    "type": "function",
                    "function": {"name": sample["tool"], "arguments": sample["parameters"]}
                }]
            },
        ],
        "tools": tools
    }

dataset = Dataset.from_list(training_data)

dataset = dataset.map(
    lambda sample: create_conversation(sample, tools, SYSTEM_MESSAGE),
    remove_columns=dataset.features,
    batched=False
)

dataset = dataset.train_test_split(test_size=TEST_SIZE, shuffle=True, seed=42)

print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Training samples: 21
Test samples: 6


<a id='step3'></a>
## Step 3: Load Model and Tokenizer

Load the pre-trained model.
**Note**: For Function Gemma accept the Gemma terms on Hugging Face before running.

In [7]:
# Used for local notebooks execution
from huggingface_hub import notebook_login
notebook_login()

In [8]:
torch_dtype = torch.float32
if torch.cuda.is_available():
    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

<a id='step4'></a>
## Step 4: Train the Model

In [9]:
sample = dataset['train'][0]
formatted = tokenizer.apply_chat_template(
    sample["messages"],
    tools=sample["tools"],
    tokenize=False
)
print("Formatted training example:")
print(formatted[:1200] + "..." if len(formatted) > 1200 else formatted)

Formatted training example:
<|im_start|>system
You are a helpful assistant that calls tools based on user requests.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "run_notebook", "description": "Execute a Jupyter notebook as a Kubeflow pipeline run.", "parameters": {"type": "object", "properties": {"path_to_notebook": {"type": "string", "description": "Path to the Jupyter notebook to execute as a pipeline"}}, "required": ["path_to_notebook"]}}}
{"type": "function", "function": {"name": "list_pipelines", "description": "List all available Kubeflow pipelines.", "parameters": {"type": "object", "properties": {}, "required": []}}}
{"type": "function", "function": {"name": "list_runs", "description": "List all pipeline runs in Kubeflow Pipelines.", "parameters": {"type": "object", "properties": {}, "required": []}}}
</tools>

For each fun

In [10]:
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="finetuned-model",
        max_length=int(MAX_LENGTH),
        packing=False,
        num_train_epochs=int(NUM_EPOCHS),
        per_device_train_batch_size=int(BATCH_SIZE),
        per_device_eval_batch_size=int(BATCH_SIZE),    
        optim="adamw_torch",
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=float(LEARNING_RATE),
        fp16=True if model.dtype == torch.float16 else False,
        bf16=True if model.dtype == torch.bfloat16 else False,
        lr_scheduler_type="constant",
        report_to="tensorboard",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        # set these for False for more speed and less Performance
        gradient_checkpointing=SAVE_MEMORY,
    ),
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
)

print("Training configuration:")
print(f"  - Epochs: {trainer.args.num_train_epochs}")
print(f"  - Batch size: {trainer.args.per_device_train_batch_size}")
print(f"  - Learning rate: {trainer.args.learning_rate}")
print(f"  - Max length: {trainer.args.max_length}")

print("Trainer initialized! Starting training...")

train_result = trainer.train()

print(f"\nTraining completed!")
print(train_result)

eval_results = trainer.evaluate()
print(eval_results)
print(f"Evaluation loss: {eval_results['eval_loss']:.4f}")
finetuned_model = trainer.model
finetuned_model

Tokenizing train dataset:   0%|          | 0/21 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/21 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/21 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/21 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

...

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
/home/wsiqueir/Downloads/gen_ai_examples/.examples/lib64/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Training completed!
TrainOutput(global_step=11, training_loss=0.787502876736901, metrics={'train_runtime': 86.9353, 'train_samples_per_second': 0.242, 'train_steps_per_second': 0.127, 'total_flos': 14570032139520.0, 'train_loss': 0.787502876736901, 'epoch': 1.0})


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
0.342668,0.321874,1,0.398786,6732.000000,0.953521


{'eval_loss': 0.3218744397163391, 'eval_entropy': 0.3987862169742584, 'eval_num_tokens': 6732.0, 'eval_mean_token_accuracy': 0.9535213708877563}
Evaluation loss: 0.3219


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [17]:
from git import Repo  # pip install gitpython

Repo.clone_from(git_url, repo_dir)



ModuleNotFoundError: No module named 'git'